# 01 — NewBench-27 scope and repo map

The cohort is **NewBench-27** = 9 discovery targets (fully computed: MD + MM-GBSA) + 18 validation targets (pre-registered, MD and MM-GBSA queued for the next compute allocation).

**Missing-data tag (concrete).** The 18 validation targets are listed in `data/external/gbsa-study/data/raw/newbench_targets.csv` (rows with `split == 'validation'`). They have no MD trajectories and no GBSA rescores yet. Rough wallclock to close the gap: ~540 productions (18 × 30) × ~24 h GPU for MD, plus ~50 s/complex × 540 × 48 GBSA combos for the rescore. The list is frozen and version-controlled; the user schedules the run when compute is available.

If those 18 land, kinase n goes 2 → 10 and protease n goes 4 → 5. Every currently-singleton family (reader, nuclear receptor, hydrolase, transporter, transferase, secreted protein, ion channel, unclassified) reaches n ≥ 2. Until then, every family-stratified claim in NB 30 is pilot-level.

This notebook:

1. **Section A** — scope table + a heatmap of computed vs pre-registered.
2. **Section B** — maps every research question in the repo to the notebook slot that answers it.

It's the entry point after `README.md` and `STUDY_DESIGN.md`.

_(Notebook auto-generated. Self-contained: exports figures to `figures/01_scope_and_map_figK.png`.)_


> **Reader guide — where this notebook sits in the study.**
>
> **Part of:** *Orientation section* — no experiment; the reader's entry point into the study.
>
> **Question this NB answers:** *what is the NewBench-27 cohort (discovery-9 + validation-18),
> which target × ligand pairs have MD + MM-GBSA computed, and which research question maps to
> which downstream notebook?*
>
> **Method:** loads the pre-registered target list + metadata + GBSA-availability heatmap.
>
> **Reproducibility contract:** only reads `data/raw/reference/ohds_metadata.csv` and
> `data/raw/reference/ohds_newbench_targets.csv`. Regenerable from a fresh clone.
>
> **Environment:** requires `pip install -e .` from the repo root.

In [ ]:
# --- notebook preamble ---
NB_STEM = "01_scope_and_map"

import sys, os
from pathlib import Path

# Make the in-repo src package importable without an install
# find repo root robustly (walks up until pyproject.toml)
_repo_root = Path.cwd()
while _repo_root != _repo_root.parent and not (_repo_root / 'pyproject.toml').is_file():
    _repo_root = _repo_root.parent
sys.path.insert(0, str(_repo_root / 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

from discovery9.style import apply_style, NAVY, GOLD, GREY, GREY_DASH as GREYD, CREAM, WHITE
from discovery9.paths import ROOT, RAW, DERIVED, EXTERNAL, FIGURES, TABLES, GBSA_STUDY
apply_style()

# --- fig-capture hook (same monkey-patch as NBs 11-18) ---
_SAVED_FIGS = globals().setdefault('_SAVED_FIGS', [])
_orig_figure = plt.figure
_orig_subplots = plt.subplots
def _figure_capture(*a, **kw):
    fig = _orig_figure(*a, **kw)
    if fig not in _SAVED_FIGS:
        _SAVED_FIGS.append(fig)
    return fig
def _subplots_capture(*a, **kw):
    fig, ax = _orig_subplots(*a, **kw)
    if fig not in _SAVED_FIGS:
        _SAVED_FIGS.append(fig)
    return fig, ax
plt.figure = _figure_capture
plt.subplots = _subplots_capture

print('ROOT:', ROOT)


## Section A — NewBench-27 scope grid

Source of truth for the target list: `data/external/gbsa-study/data/raw/newbench_targets.csv`.

Discovery (9) = MD trajectories computed **and** MM-GBSA rescored.
Validation (18) = pre-registered, queued for the next compute allocation. See the opening cell for wallclock and owner.


In [ ]:
# Load the master target list
targets_csv = GBSA_STUDY / 'data' / 'raw' / 'newbench_targets.csv'
assert targets_csv.exists(), f'{targets_csv} missing — check the gbsa-study symlink'
targets = pd.read_csv(targets_csv)

# Discovery-9 = the 9 targets with computed MD trajectories AND GBSA rescores.
DISCOVERY_9 = {'2XU3', '3I06', '4A5S', '4L7G', '4QB3', '5HU9', '8ELC', '9D9I', '9SI4'}

targets['MD_available']   = targets['pdb'].isin(DISCOVERY_9)
targets['GBSA_available'] = targets['pdb'].isin(DISCOVERY_9)

# Sort by (split, family, pdb) — discovery first, then validation.
split_order = pd.CategoricalDtype(['discovery', 'validation'], ordered=True)
targets['split'] = targets['split'].astype(split_order)
targets = targets.sort_values(['split', 'family', 'pdb']).reset_index(drop=True)

n_discovery  = int(targets['MD_available'].sum())
n_validation = int((~targets['MD_available']).sum())
print(f'NewBench-27: {len(targets)} targets  =  {n_discovery} discovery (computed)  +  {n_validation} validation (pre-registered)')

# Display the full grid
grid_cols = ['pdb', 'family', 'chembl_id', 'uniprot', 'crystal_ligand',
             'crystal_ligand_rscc', 'resolution_A', 'weakest_Ki_nM', 'split',
             'MD_available', 'GBSA_available']
grid = targets[grid_cols].copy()
print()
print(grid.to_string(index=False))


In [ ]:
# ---------- Figure 1: scope grid rendered as a table figure ----------
fig, ax = plt.subplots(figsize=(13.5, 0.42 * len(grid) + 1.3))
ax.axis('off')

col_labels = ['pdb', 'family', 'chembl_id', 'uniprot', 'lig',
              'RSCC', 'res_A', 'Ki_nM', 'split', 'MD', 'GBSA']
cell_text = []
for _, r in grid.iterrows():
    cell_text.append([
        str(r['pdb']), str(r['family']), str(r['chembl_id']), str(r['uniprot']),
        str(r['crystal_ligand']),
        str(r['crystal_ligand_rscc']),
        f"{r['resolution_A']}",
        f"{r['weakest_Ki_nM']:.0f}" if pd.notna(r['weakest_Ki_nM']) else '',
        str(r['split']),
        'Y' if r['MD_available']   else '-',
        'Y' if r['GBSA_available'] else '-',
    ])

# Row background colour: discovery rows get CREAM, validation rows get GREY.
row_colors = []
for _, r in grid.iterrows():
    row_colors.append([CREAM if r['split'] == 'discovery' else GREY] * len(col_labels))

tbl = ax.table(
    cellText=cell_text,
    colLabels=col_labels,
    cellColours=row_colors,
    colColours=[NAVY] * len(col_labels),
    loc='center', cellLoc='center', colLoc='center',
)
tbl.auto_set_font_size(False)
tbl.set_fontsize(9)
tbl.scale(1.0, 1.2)

# Header row text colour = white on navy
for j in range(len(col_labels)):
    cell = tbl[(0, j)]
    cell.get_text().set_color(WHITE)
    cell.get_text().set_fontweight('bold')

title = (f'NewBench-27 scope grid  ·  {n_discovery} discovery (computed) + '
         f'{n_validation} validation (pre-registered, not yet computed)')
ax.set_title(title, color=NAVY, fontweight='bold', pad=12)
plt.tight_layout()


In [ ]:
# ---------- Figure 2: 27x3 status heatmap ----------
# Columns: has-MD, has-GBSA, in-hardened-claim-b.
# Discovery-9 are all in hardened_claim_b.csv (the 4A5S recovery + iter-4 pipeline).
targets['in_hardened_claim_b'] = targets['MD_available']  # same 9 discovery targets

mat_cols = ['MD_available', 'GBSA_available', 'in_hardened_claim_b']
mat_labels = ['has MD', 'has GBSA', 'in hardened_claim_b']
M = targets[mat_cols].astype(int).to_numpy()

# IDIS palette: 0 -> GREY (missing), 1 -> NAVY (present)
cmap = ListedColormap([GREY, NAVY])

fig, ax = plt.subplots(figsize=(6.4, 0.32 * len(targets) + 1.4))
im = ax.imshow(M, aspect='auto', cmap=cmap, vmin=0, vmax=1)

ax.set_xticks(range(len(mat_cols)))
ax.set_xticklabels(mat_labels, rotation=25, ha='right', color=NAVY)
ax.set_yticks(range(len(targets)))
ytick_labels = [f"{r.pdb}  ({r.split[:3]})" for r in targets.itertuples()]
ax.set_yticklabels(ytick_labels, fontsize=8, color=NAVY)

# Annotate each cell with Y / '-'
for i in range(M.shape[0]):
    for j in range(M.shape[1]):
        v = M[i, j]
        ax.text(j, i, 'Y' if v == 1 else '-',
                ha='center', va='center',
                color=CREAM if v == 1 else NAVY,
                fontsize=9, fontweight='bold')

# Horizontal divider between discovery and validation blocks
n_disc = int(targets['MD_available'].sum())
ax.axhline(n_disc - 0.5, color=GOLD, lw=1.6)

ax.set_title('NewBench-27 status heatmap  ·  NAVY = present, GREY = pre-registered / not yet computed\n'
             'GOLD line separates discovery (top) from validation (bottom)',
             color=NAVY, fontweight='bold', pad=10)

ax.tick_params(axis='x', which='both', bottom=False, top=False)
ax.tick_params(axis='y', which='both', left=False, right=False)
for spine in ax.spines.values():
    spine.set_edgecolor(NAVY)
plt.tight_layout()


## Section B — research-question map (flat 00..30 slots)

### How to read this repo

1. **`README.md`** — headline result and repo layout.
2. **`STUDY_DESIGN.md`** — cohort, protocol, cross-validation, scope.
3. **`notebooks/00_data_catalog.ipynb`** — grep-able artifact index.
4. **`notebooks/01_scope_and_map.ipynb`** (this notebook) — scope grid + research-question map.
5. **`notebooks/{02..13}_*.ipynb`** — Act 1 setup + Act 2 data-side experiments (MD-config provenance, coverage, docking + GBSA baselines, physics DOE, GROMACS-config screen, temporal convergence, selection correction).
6. **`notebooks/{14..21}_*.ipynb`** — Act 3 MD features on the discovery-9 (deep dive → correlation heatmap → actives-vs-decoys z-scores).
7. **`notebooks/{22..30}_*.ipynb`** — Act 4 ranking + verdict (BEDROC baselines, ML combo selection, single-feature BEDROC, rank fusion, family stratification).
8. **`reproduce/`** — every derived table has a script; `tables/` holds finished panel-level outputs.

### Research-question map

| Research question | NB slot(s) that answer it | Reconciliation note |
|---|---|---|
| How much of the 270 × 48 GBSA-combo grid is covered? | `04_coverage`, `03_dataset_overview` (270 MD side) | complementary — different granularity |
| Docking vs GBSA baseline correlation? | `06_gbsa_vs_docking`, `22_bedroc_baselines`, `28_single_feature_bedroc` | reconciled in `data/derived/canonical_baselines.csv` |
| sp-config selection correction? | `13_selection_correction`, `23_ml_combo_selection`, `24_full_factorial_ml`, `25_deep_research_verdict` | selection-correction is post-hoc; ML variants are LOTO |
| Docking-only baseline? | `05_docking_baseline`, `22_bedroc_baselines` | canonical value in `data/derived/canonical_baselines.csv` |
| GBSA physics-factor importance? | `07_physics_importance` | upstream-only pre-flatten |
| MD wallclock throughput? | `11_timestep_throughput` | upstream-only pre-flatten |
| MD 2 vs 4 fs stability? | `10_timestep_stability` | upstream-only pre-flatten |
| Taguchi L27 GROMACS-config screening? | `09_study2_gromacs_screening` | upstream-only pre-flatten |
| DOE-toolkit on both factorials? | `08_doe_analysis` | the DOE showcase |
| Temporal convergence per target? | `12_temporal_convergence`, `26_gbsa_param_vs_md` (uses settled features) | one tells us when it settles, the other uses-settled |
| Per-target ML combo picker beats GBSA-locked? | `23_ml_combo_selection`, `24_full_factorial_ml`, `25_deep_research_verdict` | NB 25 holds the verdict |
| MD feature standalone ranks ligand actives? | `27_ligand_chem_gbsa_surrogate`, `28_single_feature_bedroc`, `29_rank_fusion_deployable` | Claim B (retracted) |
| Family stratification? | `30_family_stratification` | see small-n caveat there |

### Cross-reference tables

- `data/derived/canonical_baselines.csv` — single source of truth for the docking / GBSA-locked panel BEDROC used across the whole tree.
- `tables/panel_bedroc_summary.csv` — headline panel BEDROC per ranker.
- `tables/rank_fusion_sweep.csv` — K-fusion sweep (Claim B retraction evidence).


In [ ]:
# --- export every figure produced in this notebook (same pattern as NBs 11-18) ---
try:
    FIGURES.mkdir(parents=True, exist_ok=True)
except NameError:
    from discovery9.paths import FIGURES
    FIGURES.mkdir(parents=True, exist_ok=True)
try:
    _cream = CREAM
except NameError:
    from discovery9.style import CREAM as _cream
figs = list(globals().get('_SAVED_FIGS', []))
for num in plt.get_fignums():
    f = plt.figure(num)
    if f not in figs:
        figs.append(f)
saved = []
for i, fig in enumerate(figs, start=1):
    out = FIGURES / f"{NB_STEM}_fig{i}.png"
    try:
        fig.savefig(out, bbox_inches='tight', dpi=300, facecolor=_cream)
    except Exception as e:
        print(f'  WARN: failed to save fig{i}: {e}')
        continue
    saved.append(str(out.name))
print(f'saved {len(saved)} figures:')
for s in saved:
    print(' ', s)
